# Notebook 00: System Architecture -- End-to-End Walkthrough

## Project: LLM-Distilled IAB Content Classification Pipeline

This is the successor to [nlp-classifier-realtime](https://github.com/nbatra/nlp-classifier-realtime), which used TF-IDF + Calibrated Logistic Regression. This project explores whether LLM knowledge distillation and semantic embeddings can outperform that baseline -- and discovers that the answer is nuanced.

### Purpose of This Notebook

This notebook is a **reference document** -- no code is executed here. It explains the complete system architecture for classifying website domains into **IAB Content Taxonomy Tier-1** (27 categories) using a two-LLM pipeline: one LLM corrects labels and generates keywords, another produces soft probability distributions for distillation.

### The Core Idea

**In plain English:** We ask a smart (but slow and expensive) AI model to categorize 100,000 websites and describe them with English keywords. Then we compare four approaches to using those labels at production speed: TF-IDF on the keywords, an MLP trained on embeddings with soft-label distillation, and a full transformer fine-tuned end-to-end.

**Technically:** We use a two-LLM approach:
1. **Sonnet 4** corrects the noisy Kaggle ground-truth labels (49.7% were wrong) and generates English keywords for all 97K domains
2. **Opus 4.6** produces soft probability distributions over 27 categories for 40K+ domains -- the distillation signal

Four downstream models are compared:
- **TF-IDF + LinearSVC** (91.6% top-1, 0.02ms inference) -- exploits keywords as direct bag-of-words features
- **E5-small-v2 + MLP** with KL distillation (84.9% top-1, ~1ms inference) -- learns from soft teacher labels
- **ModernBERT fine-tuning** (150M params, ~10ms inference) -- full encoder fine-tuning on corrected labels
- **Logistic Regression / SGD baselines** (89.7-90.4% top-1) -- linear models on TF-IDF features

### The Surprising Result

TF-IDF + LinearSVC wins. When Sonnet generates keywords like "online store, electronics, gadgets," those tokens ARE the classification expressed in natural language. TF-IDF converts them directly into high-IDF discriminative features. Dense embeddings lose this lexical precision by compressing everything into 384 dimensions.

This does NOT mean TF-IDF is generally better. It means that when the upstream LLM has already done the semantic heavy lifting, the downstream model's job is trivial -- and simpler models exploit trivial signals more directly.

### Why Rebuild the [Previous Project](https://github.com/nbatra/nlp-classifier-realtime)?

The previous project uses TF-IDF + Calibrated Logistic Regression on raw domain metadata. This works but has fundamental limitations:

| Limitation | TF-IDF Approach (previous project) | This Project |
|---|---|---|
| **Label quality** | Trained on Kaggle labels (49.7% wrong) | LLM-corrected labels |
| **Feature quality** | Raw domain names + crawled descriptions | LLM-generated English keywords |
| **Vocabulary dependence** | Unknown words get zero weight | Keywords are in controlled English vocabulary |
| **Semantic understanding** | "car" and "automobile" are unrelated | Keywords normalize synonyms |
| **Multi-language** | Needs separate vocabulary per language | Sonnet generates English keywords for all domains |

The key finding: fixing the input (labels + keywords) matters far more than changing the model. TF-IDF with LLM-corrected data achieves 91.6% -- vastly better than any model trained on noisy data.

### Models Implemented and Results

| # | Approach | Val Top-1 | Val Top-3 | Speed | Best For |
|---|---|---|---|---|---|
| 1 | **TF-IDF + LinearSVC** | **91.6%** | **99.3%** | 0.02 ms | Production (fastest, best accuracy) |
| 2 | **MLP + KL Distillation** | 84.9% | 98.3% | ~1 ms | When embeddings are already cached |
| 3 | **ModernBERT Fine-tune** | (in progress) | -- | ~10 ms | Quality ceiling exploration |
| 4 | TF-IDF + Logistic Regression | 90.4% | 99.4% | 0.02 ms | Alternative linear baseline |

### Sections

1. The Problem -- What is IAB classification and why it matters
2. The Full Pipeline -- From raw domains to production classifier (two-LLM architecture)
3. Knowledge Distillation -- How the teacher-student framework works
4. Concrete Example -- "espn.com" through each approach
5. Extreme Multi-Label Strategy -- Handling 700 classes (Tier-2 expansion)
6. Model Architecture Details -- Each approach in depth
7. Latency and Cost Budget -- Production economics
8. Notebook Dependency Graph -- What produces and consumes what
9. FAQ

## Section 1: The Problem

### What is IAB Content Taxonomy?

The **Interactive Advertising Bureau (IAB)** defines a standard taxonomy for categorizing web content. Version 2.2 has approximately **700 categories** organized in a two-level hierarchy:

```
Tier-1 (23 top-level categories):
  IAB-1:  Arts & Entertainment
  IAB-2:  Automotive
  IAB-3:  Business
  IAB-4:  Careers
  IAB-5:  Education
  IAB-6:  Family & Parenting
  IAB-7:  Health & Fitness
  IAB-8:  Food & Drink
  IAB-9:  Hobbies & Interests
  IAB-10: Home & Garden
  IAB-11: Law, Government & Politics
  IAB-12: News
  IAB-13: Personal Finance
  IAB-14: Society
  IAB-15: Science
  IAB-16: Pets
  IAB-17: Sports
  IAB-18: Style & Fashion
  IAB-19: Technology & Computing
  IAB-20: Travel
  IAB-21: Real Estate
  IAB-22: Shopping
  IAB-23: Religion & Spirituality

Tier-2 (nested under each Tier-1, ~30 per category):
  IAB-17:  Sports
    IAB-17-1:  Auto Racing
    IAB-17-2:  Baseball
    IAB-17-3:  Bicycling
    IAB-17-4:  Bodybuilding
    IAB-17-5:  Boxing
    IAB-17-6:  Canoeing/Kayaking
    IAB-17-7:  Cheerleading
    IAB-17-8:  Climbing
    IAB-17-9:  Cricket
    IAB-17-10: Figure Skating
    IAB-17-11: Fly Fishing
    IAB-17-12: Football
    ...         (~35 total sub-categories under Sports)
```

### Why Does This Matter?

**Layman's explanation:** Every time you visit a website, advertisers are bidding in real-time (under 100ms) to show you an ad. If the system knows that espn.com is a "Sports" website and you have been visiting sports sites all week, a Nike ad is worth more to show you than a random ad. The classification of websites into categories is what makes this targeting possible.

**Technical explanation:** In programmatic advertising, domain classification feeds directly into:
- **Bid enrichment**: Real-time bid requests include the page URL. The domain's IAB categories are looked up and attached to the bid request, enabling category-targeted campaigns.
- **User profiling**: Accumulate per-user IAB interest vectors from browsing history. A user visiting travel sites gets scored high on IAB-20 (Travel).
- **Brand safety**: Advertisers exclude certain categories (IAB-25: Non-Standard Content, IAB-26: Illegal Content).
- **Contextual targeting**: Post-cookie world relies on page content classification rather than user tracking.

### Why is This Hard?

1. **Scale**: 200K+ domains need classification, new domains appear daily
2. **Multi-label**: A domain like "nytimes.com/travel" is both News AND Travel
3. **Extreme class count**: ~700 fine-grained categories, many with very few training examples
4. **Ambiguity**: Is "reddit.com" Technology? Social Media? It depends on the subreddit
5. **Speed**: Classification must complete in <2ms for real-time bid enrichment
6. **Label scarcity**: Getting human annotations for 700 categories across 100K domains is prohibitively expensive
7. **Label noise**: Available datasets (like Kaggle) have ~49% incorrect labels -- training on noisy ground truth caps model performance

### How the LLM Solves the Labeling Problem

**The key insight**: Large language models like Claude already "know" the IAB taxonomy. They have seen it in training data, understand the category definitions, and can classify domains with high accuracy -- zero-shot, with no task-specific training. The problem is that calling an LLM for every bid request (100ms+ latency, $0.01+/call) is impossible at AdTech scale (millions of QPS). 

**The solution**: Use the LLM once (offline) to label everything, then train a fast student model that memorizes the LLM's answers. The student runs at TF-IDF speed but with LLM-quality labels.

**The bonus**: The LLM can also FIX existing noisy labels. In our Kaggle dataset, 49% of tier-1 labels are incorrect. Sonnet 4 corrects these before we train, removing the label-noise ceiling that limited the [previous project](https://github.com/nbatra/nlp-classifier-realtime).

## Section 2: The Full Pipeline

The system has two major phases: an offline phase (run once or periodically) that builds all models and lookup tables, and an online phase (real-time) that serves predictions.

### The Two-LLM Architecture

A critical finding from v1: Kaggle ground-truth labels disagree with Opus on 49% of domains (many obviously wrong -- porn sites labeled "Arts & Entertainment", sports sites labeled "News"). Training a student against these noisy labels caps accuracy at ~45%, regardless of teacher coverage or model capacity.

The solution uses two LLMs with distinct roles:

```
+==============================================================================+
|  SONNET 4 (Label Corrector + Keyword Generator)                              |
|=======================================|=======================================|
|  Purpose: Fix ground-truth labels     |  OPUS 4.6 (Teacher)                  |
|  + generate English keywords          |=======================================|
|  Coverage: ALL 97K domains            |  Purpose: Soft probability labels    |
|  Output per domain:                   |  Coverage: 40K+ stratified subset    |
|    - Single best tier1 category       |  Output per domain:                  |
|    - 5-10 English keywords            |    - Probabilities over all 27 cats  |
|  Speed: ~7 domains/sec (10 workers)   |  Speed: ~2.5 domains/sec (5 workers) |
|  Cost: Lower (Sonnet pricing)         |  Cost: Higher (Opus pricing)         |
|  Why Sonnet: straightforward task,    |  Why Opus: nuanced judgment needed   |
|    just needs correct classification  |    for probability calibration       |
+==============================================================================+
```

### Why Two LLMs?

Different tasks need different capabilities:
- **Sonnet 4**: Label correction and keyword extraction is a straightforward task. "Is this domain about Sports or Entertainment?" Sonnet is fast and cheap, enabling coverage of all 97K domains. It also generates English keywords that become the most powerful feature in the pipeline.
- **Opus 4.6**: Producing calibrated soft probability distributions across 27 categories requires nuanced judgment. "This is 95% Sports but also 15% Entertainment because of their celebrity coverage." Opus excels at this but is slower and more expensive, so we use it on a stratified subset.

The v1 experiment proved this division matters: adding more Opus labels (6.5K to 16K to 40K) improved accuracy, but fixing the ground truth (Sonnet's job) was the breakthrough that enabled the v1-to-v2 jump from 45.1% to 84.9%.

### Offline Phase (Training Pipeline)

```
+=============================================================================+
|                    OFFLINE PHASE (Training Pipeline)                         |
+=============================================================================+
|
|  Step 0: Data Acquisition (v1/01)
|  ----------------------------------
|  Kaggle Dataset (bpmtips/websiteiabcategorization): 98K domains
|    --> domain names + IAB category labels (noisy ground truth)
|    --> Train/Val/Test split (80/10/10, stratified)
|
|  Step 1: Data Correction with Sonnet 4 (v2/01)
|  ------------------------------------------------
|  For each of the 97K unique domains:
|    Input:  domain name + title + description (from Kaggle)
|    Output: {"category": "Sports", "keywords": "football, scores, NFL, ..."}
|
|  Result: 49.7% of labels changed. Shopping +104%, Sensitive Subjects 99% wrong.
|  The keywords become the TEXT FEATURES for all downstream models.
|  Saved to: data/corrected/{train,val,test}.parquet with columns:
|    - tier1 (corrected by Sonnet)
|    - tier1_kaggle (original, preserved for comparison)
|    - keywords (Sonnet-generated English keywords)
|    - text = "domain | title | keywords" (enriched representation)
|
|  Step 2: Teacher Labeling with Opus 4.6 (v2/03)
|  -------------------------------------------------
|  For a stratified subset (40,696 domains covering all 27 categories):
|    Input:  domain name + metadata
|    Prompt: "Classify into IAB Tier-1 categories with confidence scores"
|    Output: {"Sports": 0.95, "Arts & Entertainment": 0.15, ...}
|
|  Opus produces SOFT LABELS (probability distributions over 27 classes).
|  Mean confidence: 0.782. Mean 2.9 categories per domain.
|  32,919 of these fall in the training set (42.4% coverage).
|
|  Step 3: Embedding Generation (v2/03)
|  ---------------------------------------
|  E5-small-v2 encodes the enriched text ("domain | title | keywords"):
|    "espn.com | ESPN - Sports | football, NBA, MLB, scores"
|      --> E5-small-v2 --> [0.23, -0.45, 0.12, ..., 0.67] (384 floats)
|
|  Speed: 1,184 domains/sec on Apple Silicon MPS (2x faster than v1
|  because English keywords are shorter than multilingual descriptions).
|  Output: .npy files (97K x 384 matrix, ~143 MB)
|
|  Step 4: Model Training (v2/04, v2/05, v2/06)
|  -----------------------------------------------
|  Three parallel paths:
|
|  PATH A: TF-IDF + LinearSVC (v2/06) -- THE WINNER
|    TF-IDF(30K features, uni+bigrams, sublinear_tf) --> LinearSVC(class_weight='balanced')
|    Result: 91.6% top-1, 99.3% top-3, trains in 12.5 seconds
|
|  PATH B: MLP Distillation (v2/04)
|    Input:  pre-computed E5 embedding (384-dim)
|    Target: Opus soft labels (27-dim) + corrected hard labels
|    Loss:   0.7 * KL(student, teacher_soft) + 0.3 * weighted_BCE(student, hard)
|    Architecture: 384 -> 512 -> BN -> GELU -> Drop(0.3)
|                      -> 256 -> BN -> GELU -> Drop(0.3) -> 27
|    Result: 84.9% top-1, 98.3% top-3, 337K params, trains in 25 seconds
|
|  PATH C: ModernBERT Fine-tuning (v2/05)
|    Full 150M-param encoder fine-tuning on corrected labels
|    WeightedTrainer with class_weight='balanced', 3 epochs, lr=2e-5
|    Result: (in progress, expected 90-95% based on corrected label quality)
|
+=============================================================================+
```

### Data Flow for a Single Domain (v2 Pipeline)

```
"espn.com" (from Kaggle, labeled "Arts & Entertainment" -- WRONG)
     |
     |--- Step 1: Sonnet 4 corrects --> tier1="Sports"
     |                                  keywords="football, NBA, MLB, scores, highlights"
     |
     |--- Step 2: Opus 4.6 produces --> soft labels: {Sports: 0.95, Entertainment: 0.15, ...}
     |
     |--- Step 3: E5-small encodes  --> embedding: [0.23, -0.45, ..., 0.67] (384 floats)
     |              (using enriched text: "espn.com | ESPN | football, NBA, MLB, scores")
     |
     |--- Step 4a: TF-IDF vectorizes text --> sparse 30K-dim vector
     |              "football" IDF=high, "NBA" IDF=high --> strong Sports features
     |              LinearSVC classifies --> "Sports" (0.02ms)
     |
     |--- Step 4b: MLP predicts from embedding
     |              Trained on Opus soft labels via KL divergence
     |              MLP(embedding) --> "Sports" (0.1ms, embedding pre-cached)
     |
     v
  AT INFERENCE TIME (production):
     Known domain?   --> Tier 1: lookup_table["espn.com"] --> instant (<0.5ms)
     Unknown domain? --> Tier 2: TF-IDF vectorize + LinearSVC classify (0.02ms)
                         OR: E5-small embed + MLP classify (~2ms)
     Low confidence? --> Tier 3: Queue for LLM (async)
```

### Online Phase (Production Inference)

```
+=============================================================================+
|                    ONLINE PHASE (Production Inference)                       |
+=============================================================================+
|
|  A bid request arrives with a domain (e.g., "espn.com")
|
|  Tier 1: Lookup Table (pre-computed, <0.5ms)
|  -------------------------------------------
|  Hash map lookup: domain --> IAB probability vector
|  Covers ~85% of bid volume (all known 97K domains)
|  Memory: 97K entries * ~600 bytes/entry = ~58MB (fits in L3 cache)
|
|  Tier 2: Real-Time Classification (~0.02ms with TF-IDF, ~2ms with MLP)
|  -----------------------------------------------------------------------
|  For unknown/new domains:
|    Option A (fastest): TF-IDF vectorize + LinearSVC predict
|      Requires: the domain's text (title + keywords -- from crawl or heuristic)
|      Latency: 0.02ms
|    Option B (no keyword dependency): E5-small embed + MLP predict
|      Requires: just the domain name (embedding handles raw text)
|      Latency: ~2ms (embedding dominates)
|
|  Tier 3: LLM Fallback (async, for quality)
|  -------------------------------------------
|  For domains where classifier confidence is very low (max_prob < 0.3):
|    - Queue for async LLM classification
|    - Serve best-effort prediction immediately
|    - Update lookup table when response arrives (~500ms-2s later)
|
+=============================================================================+
```

### Why Three Tiers?

| Tier | Latency | Coverage | Quality | Cost/Query |
|---|---|---|---|---|
| 1. Lookup table | <0.5ms | ~85% of traffic | Highest (pre-validated) | $0 |
| 2. TF-IDF classifier | 0.02ms | ~14% of traffic | 91.6% top-1 | ~$0.000001 |
| 2b. MLP (alternative) | ~2ms | ~14% of traffic | 84.9% top-1 | ~$0.000001 |
| 3. LLM fallback | 500ms+ (async) | ~1% of traffic | Highest (teacher directly) | ~$0.001 |

## Section 3: Knowledge Distillation -- How It Works

### The Concept in Plain English

Imagine you hire an expert sommelier (Opus) to taste and categorize 40,000 wines. The sommelier provides detailed notes: "This wine is 95% Bordeaux-style, 15% suitable for celebration pairings." These nuanced assessments are the **soft labels**.

Separately, a faster assistant (Sonnet) corrects the wine labels in your catalog -- 49.7% of them were wrong -- and adds English tasting keywords for all 97,000 wines.

Now you train an apprentice (the student MLP) by showing them the same wines and asking them to reproduce the sommelier's assessments. The apprentice learns not just "this is a Bordeaux" (hard label) but the full distribution -- including the sommelier's uncertainty and the relationships between categories.

The surprise: a simple keyword lookup (TF-IDF) using the assistant's keywords actually outperforms the trained apprentice. The assistant's keywords were so good that they effectively pre-solved the classification.

### The Technical Framework (Actual Implementation)

```
TEACHER MODEL (Opus 4.6 via AWS Bedrock)
================================================

Input: "espn.com" + IAB Taxonomy reference (system prompt)

Opus's reasoning:
  - ESPN is a major sports network
  - Covers football, basketball, baseball, etc.
  - Has some entertainment/celebrity sports content
  - Fantasy sports section relates to gaming

Output (soft labels, 27-dim probability vector):
  {
    "Sports":              0.95,
    "Arts & Entertainment": 0.15,
    "Games":               0.08,
    "Hobbies & Leisure":   0.05,
    ... (remaining 23 categories at ~0.0)
  }

Mean confidence across 40,696 labeled domains: 0.782
Mean categories per domain (above threshold): 2.9

CRITICAL: These soft labels contain MORE information than a hard label.
  - Hard label: "Sports" (1 bit -- just the argmax)
  - Soft label: Full 27-dim distribution (rich inter-class signal)
  - The 0.15 for Entertainment tells the student about category relationships
```

```
STUDENT MODEL (E5-small-v2 + MLP, 337K trainable params)
================================================

Architecture (actual v2 implementation):
  Input: "espn.com | ESPN | football, NBA, MLB, scores"
    |
    v
  [E5-small-v2 encoder, FROZEN] --> 384-dim L2-normalized embedding
    |
    v
  [Linear(384, 512) + BatchNorm + GELU + Dropout(0.3)]
    |
    v
  [Linear(512, 256) + BatchNorm + GELU + Dropout(0.3)]
    |
    v
  [Linear(256, 27)]
    |
    v
  Output: 27-dim logits --> softmax for probabilities

Total trainable params: 337,179
Model size on disk: 1.36 MB
Training time: 25 seconds (50 epochs, batch 512, MPS)
Best epoch: 13 (early stopping on val accuracy)

Result: 84.9% top-1, 98.3% top-3, Macro-F1 = 0.814
```

### The Complete Training Procedure (Step by Step)

This is the end-to-end process of how the student model learns from the teacher:

```
STEP 1: PRE-COMPUTE TEACHER LABELS (offline, done once)
========================================================

For each of 40,696 stratified domains, call Opus via AWS Bedrock:

  Input prompt:
    System: "You are a website classification expert. Classify into IAB Tier-1
             categories with confidence scores (0.0-1.0). Return only JSON."
    User:   "Domain: espn.com | Title: ESPN | Keywords: football, NBA, MLB"

  Opus returns:
    {"Sports": 0.95, "Arts & Entertainment": 0.15, "Games": 0.08}

  We convert this to a 27-dim vector:
    teacher_soft[i] = confidence for category i, else 0.0
    [0.0, 0.0, 0.0, ..., 0.95, ..., 0.15, ..., 0.08, ...]
     Adult  A&E  Autos       Sports    A&E       Games

  Saved to: data/processed/teacher_labels.parquet
    - 40,696 rows
    - Columns: domain_clean + 27 probability columns
    - 32,919 of these are in the training set (42.4% of 77,588 train domains)


STEP 2: PRE-COMPUTE EMBEDDINGS (offline, done once)
========================================================

For each of 77,588 train domains, encode the enriched text with E5-small-v2:

  text = "espn.com | ESPN - Serving Sports Fans | football, NBA, MLB, scores"
  embedding = E5_small_v2.encode("passage: " + text)  --> 384-dim, L2-normalized

  Saved to: data/corrected/embeddings/train_embeddings.npy  (77,588 x 384 matrix)
  Saved to: data/corrected/embeddings/train_domains.parquet (index mapping)

  Speed: 1,184 domains/sec on MPS. Total: ~65 seconds for all train domains.


STEP 3: PREPARE TRAINING DATA (at training time)
========================================================

  Load:
    X_train = np.load("train_embeddings.npy")           # shape: (77,588, 384)
    y_hard  = label_encoder.transform(train_df['tier1']) # shape: (77,588,) integers 0-26
    y_soft  = teacher_labels aligned by domain           # shape: (32,919, 27) floats

  Create boolean mask:
    has_teacher[i] = True if domain i has Opus soft labels (42.4% of rows)

  Compute class weights (for imbalance handling):
    weights[c] = N_total / (N_classes * N_samples_in_class_c)
    Range: 0.22 (Shopping, largest) to 55.8 (Sensitive Subjects, smallest)


STEP 4: TRAINING LOOP (the actual learning)
========================================================

  Hyperparameters:
    epochs = 50
    batch_size = 512
    learning_rate = 1e-3
    optimizer = AdamW(weight_decay=1e-4)
    scheduler = ReduceLROnPlateau(patience=5, factor=0.5)
    alpha = 0.7          (distillation weight)
    temperature = 2.0    (softens teacher distribution)
    early_stopping_patience = 10

  For each epoch (1 to 50):
    Shuffle all 77,588 training samples
    For each batch of 512 samples:

      # Get batch data
      x_batch = embeddings[batch_indices]       # (512, 384)
      y_hard_batch = hard_labels[batch_indices]  # (512,) integers

      # Forward pass through student MLP
      logits = student_mlp(x_batch)              # (512, 27)

      # CASE A: Samples WITH teacher labels (about 42.4% of batch ~ 217 samples)
      mask = has_teacher[batch_indices]
      if mask.any():
          teacher_batch = y_soft[batch_indices[mask]]  # (~217, 27)

          # Soften both distributions with temperature
          student_log_probs = log_softmax(logits[mask] / T)   # softer student
          teacher_probs = softmax(teacher_batch / T)           # softer teacher

          # KL divergence: how different is student from teacher?
          kl_loss = KL_div(student_log_probs, teacher_probs) * T^2
          # T^2 scaling ensures gradient magnitudes are independent of T

      # CASE B: ALL samples get hard label loss (anchoring)
      hard_loss = weighted_cross_entropy(logits, y_hard_batch, class_weights)

      # Combined loss
      if mask.any():
          total_loss = alpha * kl_loss + (1 - alpha) * hard_loss
      else:
          total_loss = hard_loss  # no teacher signal for this batch

      # Backward pass + update
      total_loss.backward()
      optimizer.step()
      optimizer.zero_grad()

    # End of epoch: evaluate on validation set
    val_accuracy = evaluate(student_mlp, X_val, y_val)  # top-1 accuracy
    scheduler.step(val_accuracy)

    # Early stopping check
    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        save_checkpoint(student_mlp, "student_mlp_v2_best.pt")
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= 10:
            break  # stop training, use best checkpoint

  Final result: best model at epoch 13, val accuracy = 84.9%


STEP 5: WHAT THE STUDENT LEARNED
========================================================

After training, the student MLP has internalized:

  1. DIRECT MAPPING: embedding --> category (from hard labels)
     "Domains with embeddings near cluster X tend to be Sports"

  2. INTER-CLASS STRUCTURE: from soft labels
     "Sports domains also have non-zero Entertainment and Games signal"
     "Health domains correlate with Science and Family"
     "News and Travel often co-occur (travel sections in newspapers)"

  3. UNCERTAINTY CALIBRATION: from teacher confidence levels
     "High confidence = unambiguous domain (espn.com = 0.95 Sports)"
     "Low confidence = ambiguous domain (reddit.com = 0.4 Technology, 0.3 Communities)"

  The student can now predict a full 27-dim distribution for ANY domain
  given its E5 embedding -- including domains the teacher never labeled.
```

### Visualizing the Training Signal

```
EPOCH 1: Student predictions are random (uniform ~3.7% per class)
  espn.com: student predicts {Sports: 0.04, News: 0.04, Shopping: 0.04, ...}
  teacher says:              {Sports: 0.95, Entertainment: 0.15, Games: 0.08}
  KL divergence is LARGE --> strong gradient pushes student toward teacher

EPOCH 5: Student has learned the major clusters
  espn.com: student predicts {Sports: 0.72, Entertainment: 0.08, Games: 0.05}
  teacher says:              {Sports: 0.95, Entertainment: 0.15, Games: 0.08}
  KL divergence is MEDIUM --> refining the tail of the distribution

EPOCH 13 (best): Student closely matches teacher
  espn.com: student predicts {Sports: 0.93, Entertainment: 0.12, Games: 0.06}
  teacher says:              {Sports: 0.95, Entertainment: 0.15, Games: 0.08}
  KL divergence is SMALL --> student has converged

EPOCH 20+: Overfitting begins (val accuracy drops)
  Student memorizes training set noise, early stopping prevents this
```

### The Role of Temperature (T = 2.0)

Temperature controls how much inter-class information the student receives:

```
Teacher raw output for espn.com:
  {Sports: 0.95, Entertainment: 0.15, Games: 0.08, Hobbies: 0.05, others: ~0.0}

After softmax with T=1.0 (no temperature):
  {Sports: 0.89, Entertainment: 0.04, Games: 0.02, others: ~0.00}
  Problem: The distribution is too peaked. Student only learns "this is Sports"
  and gets almost no gradient for the secondary categories.

After softmax with T=2.0 (our setting):
  {Sports: 0.52, Entertainment: 0.18, Games: 0.12, Hobbies: 0.09, others: 0.02}
  Better: The distribution is spread out. Student receives meaningful gradient
  for Entertainment (0.18) and Games (0.12), learning the inter-class structure.

After softmax with T=5.0 (too soft):
  {Sports: 0.25, Entertainment: 0.18, Games: 0.16, Hobbies: 0.14, others: 0.05}
  Problem: Everything looks equally likely. The primary signal (Sports) is diluted.
  Student learns weak, uniform relationships instead of clear category boundaries.

T=2.0 is the sweet spot: preserves the primary category signal while revealing
enough of the secondary structure for the student to learn inter-class relationships.
```

### The Distillation Loss in Mathematical Detail

```
Given a training domain:
  x = E5 embedding (384-dim, pre-computed)
  y_hard = corrected hard label (27-dim one-hot, from Sonnet)
  y_soft = teacher soft label (27-dim distribution, from Opus) -- if available

  student_logits = MLP(x)  --> 27-dim raw logits

CASE 1: Domain has Opus soft labels (42.4% of training data):
  L = alpha * KL_div(log_softmax(student_logits/T), softmax(y_soft/T)) * T^2
    + (1 - alpha) * weighted_BCE(sigmoid(student_logits), y_hard)

  Where:
    alpha = 0.7 (70% distillation, 30% hard label anchoring)
    T = 2.0 (temperature -- softens distributions to reveal inter-class structure)
    weighted_BCE uses class weights (253x imbalance: Shopping vs Sensitive Subjects)

  Why T^2 scaling?
    When T > 1, softmax outputs are compressed toward uniform.
    The gradients from KL divergence are proportionally smaller (by ~1/T^2).
    Multiplying by T^2 restores the gradient magnitude to match the hard-label loss.
    Without this, the distillation term would be too weak relative to BCE.

CASE 2: Domain has only hard labels (57.6% of training data):
  L = weighted_BCE(sigmoid(student_logits), y_hard)

This mixed training allows the student to learn from:
  - ALL 78K training domains via corrected hard labels
  - 33K training domains via rich Opus soft distributions
```

### Why Soft Labels Are Superior to Hard Labels (With Actual Numbers)

The v1 vs v2 comparison proves this empirically:

| Configuration | Teacher Coverage | Label Quality | Top-1 Accuracy |
|---|---|---|---|
| v1: Hard labels only (noisy Kaggle) | 0% soft | 50.9% correct | ~40% |
| v1: Hard + soft (noisy Kaggle + 6.5K Opus) | 8.4% soft | 50.9% correct | 45.1% |
| v2: Hard labels only (corrected) | 0% soft | ~82% correct | ~78% (estimated) |
| v2: Hard + soft (corrected + 33K Opus) | 42.4% soft | ~82% correct | **84.9%** |

The soft labels add ~7pp over hard-only with corrected labels. But the biggest gain came from fixing the labels themselves (+40pp).

### What the Student Learns from Soft Labels

Consider three domains and their actual teacher outputs:

```
Domain              | Hard Label      | Soft Labels (Opus)
--------------------+-----------------+-------------------------------------------
espn.com            | Sports          | Sports: 0.95, Entertainment: 0.15, Games: 0.08
nytimes.com/travel  | News            | Travel: 0.85, News: 0.70, Society: 0.30
webmd.com           | Health          | Health: 0.95, Science: 0.40, Family: 0.25
```

**What the student learns from hard labels:** "espn.com = Sports, everything else = 0"

**What the student learns from soft labels:**
- ESPN has a secondary entertainment angle (Sports-Entertainment correlation)
- NYTimes travel section is more Travel than News (content > brand)
- WebMD relates to Science and Family (Health-Science-Family cluster)

The soft labels encode **inter-class relationships** that hard labels discard.

### Why TF-IDF Beat the Distilled Student

The distilled student (84.9%) is impressive but loses to TF-IDF (91.6%) for a specific reason:

```
TF-IDF pipeline:
  "espn.com | ESPN | football, NBA, MLB, scores"
  --> TF-IDF extracts: "football"=0.42, "NBA"=0.38, "scores"=0.31, "MLB"=0.35
  --> These are DIRECTLY discriminative for Sports (high IDF, rare in other categories)
  --> LinearSVC draws a clean hyperplane in this sparse space

MLP pipeline:
  "espn.com | ESPN | football, NBA, MLB, scores"
  --> E5-small maps to 384-dim dense vector where "football" and "soccer" are nearby
  --> The MLP must learn to separate Sports from similar categories in this blurred space
  --> Information is COMPRESSED -- "football" is no longer a discrete high-value feature
```

The E5 embedding is designed for semantic similarity, not classification. It groups related concepts together, which helps generalization but hurts when the original tokens were already perfectly discriminative.

### Why Not Just Use the LLM Directly?

| Factor | LLM Direct | TF-IDF Student | MLP Student |
|---|---|---|---|
| Latency | 500-2000ms | 0.02ms | ~2ms |
| Cost per query | ~$0.001 | ~$0.000001 | ~$0.000001 |
| Throughput | ~100 QPS (rate limited) | 1M+ QPS | 100K+ QPS |
| Offline capable | No (needs API) | Yes | Yes |
| Deterministic | No (temperature > 0) | Yes | Yes |
| Quality | Ceiling (by definition) | 91.6% top-1 | 84.9% top-1 |

At AdTech scale (millions of QPS), the student models are the only viable production option.

## Section 4: Concrete Example -- "espn.com" Through Each Approach

Let us trace a single domain through all five approaches to make the differences concrete.

---

### Approach 1: TF-IDF + Logistic Regression (Baseline)

**Input preparation:**
```
domain_text = "espn sports entertainment football basketball baseball"
  (domain name + crawled meta description / keywords)
```

**Vectorization:**
```
TF-IDF vectorizer (20,000 features, unigrams + bigrams):
  "espn":        0.42  (high IDF -- rare term, identifies this domain)
  "sports":      0.18  (low IDF -- common across many domains)
  "football":    0.31  (medium IDF)
  "basketball":  0.33  (medium IDF)
  "baseball":    0.35  (medium IDF)
  ... (19,995 other features = 0.0)

Result: sparse 20,000-dim vector with ~10-50 non-zero entries
```

**Classification:**
```
OneVsRest Logistic Regression (700 binary classifiers):
  Classifier_IAB17(sports_vector) = sigmoid(w . x + b) = 0.91
  Classifier_IAB17_12(sports_vector) = sigmoid(w . x + b) = 0.73
  Classifier_IAB1(sports_vector) = sigmoid(w . x + b) = 0.12
  ... (697 other classifiers, most output ~0.01)

Output: {IAB-17: 0.91, IAB-17-12: 0.73, IAB-1: 0.12, ...}
Inference time: ~0.2ms
```

**Limitations visible here:**
- If the domain text said "athletics" instead of "sports", the model might miss it (OOV or weak weight)
- "ESPN" as a brand name carries information only if seen in training data
- No understanding that football, basketball, baseball are ALL sports (treats them as independent features)

---

### Approach 2: LLM Distillation (Primary Approach)

**Step A -- Teacher labeling (offline, done once):**
```
Prompt to Claude:
  System: [Full IAB Taxonomy v2.2 -- cached via prompt_caching]
          You are a domain classification expert. Classify the given domain
          into IAB Content Taxonomy v2.2 categories.

  User: Domain: espn.com
        Return top categories with confidence (0.0-1.0).
        JSON format: {"categories": [{"id": "IAB-XX", "confidence": 0.X}, ...]}

Claude's response:
  {
    "categories": [
      {"id": "IAB-17",    "name": "Sports",              "confidence": 0.98},
      {"id": "IAB-17-12", "name": "Football",            "confidence": 0.85},
      {"id": "IAB-17-2",  "name": "Baseball",            "confidence": 0.80},
      {"id": "IAB-17-1",  "name": "Auto Racing",         "confidence": 0.40},
      {"id": "IAB-1",     "name": "Arts & Entertainment","confidence": 0.25},
      {"id": "IAB-9",     "name": "Hobbies & Interests", "confidence": 0.20}
    ]
  }
```

**Step B -- Student training (offline):**
```
The student sees 98,000 such (domain, soft_label) pairs and learns to predict
the teacher's distribution from domain embeddings alone.
```

**Step C -- Student inference (online, production):**
```
Input: "espn.com"
  --> E5-small tokenizer: [101, 3802, 2078, 1012, 4012, 102]  (subword tokens)
  --> E5-small encoder:   384-dim dense embedding
                          [0.23, -0.45, 0.12, ..., 0.67]
  --> MLP head:           Linear(384,256) -> ReLU -> Linear(256,700) -> Sigmoid
  --> Output:             {IAB-17: 0.96, IAB-17-12: 0.81, IAB-17-2: 0.77, ...}

Inference time: ~2-3ms (embedding) + ~0.1ms (MLP) = ~3ms total
```

**Why this works better than TF-IDF:**
- E5-small KNOWS that "ESPN" is a sports brand (from pre-training on web text)
- The embedding captures semantic meaning, not just word overlap
- Soft labels teach inter-category relationships (Sports correlates with Entertainment)

---

### Approach 3: Embedding + Classification Head

**Difference from Approach 2:** Instead of Claude's soft labels, this trains on the Kaggle dataset's hard labels directly. No LLM involved.

```
Input: "espn.com"
  --> E5-base encoder:    768-dim dense embedding
  --> Classification head: Linear(768,256) -> ReLU -> Linear(256,700) -> Sigmoid
  --> Output:             {IAB-17: 0.89, IAB-17-12: 0.68, ...}

Training loss: BCE(predictions, kaggle_hard_labels)
  - Only learns from hard labels (less rich signal than soft labels)
  - But no LLM cost required

Inference time: ~5ms (larger encoder) + ~0.1ms (MLP) = ~5ms total
```

---

### Approach 4: SetFit (Few-Shot, Tier-1 Only)

**How SetFit works:**
```
Training (only needs 8-16 examples per class):
  1. Start with pre-trained sentence-transformer (e.g., all-MiniLM-L6-v2)
  2. Generate contrastive pairs:
     Positive: (espn.com, nfl.com)         -- both Sports
     Negative: (espn.com, zillow.com)      -- Sports vs Real Estate
  3. Fine-tune with contrastive loss: push positives together, negatives apart
  4. Train a logistic regression head on the fine-tuned embeddings

Inference:
  Input: "espn.com"
    --> Fine-tuned MiniLM: 384-dim embedding (now sports-aware)
    --> Logistic head:     {Sports: 0.94, Entertainment: 0.15, ...}

Inference time: ~2ms
```

**Limitation:** SetFit scales poorly beyond ~50-100 classes (contrastive pair generation explodes combinatorially). We use it only for the 23 Tier-1 categories as a rapid prototype.

---

### Approach 5: Fine-tuned ModernBERT

**Full encoder fine-tuning (highest quality, slowest training):**
```
Input: "espn.com"
  --> ModernBERT tokenizer: subword tokens
  --> ModernBERT encoder (150M params, ALL weights updated during training):
      12 Transformer layers with RoPE + Flash Attention
      --> 768-dim [CLS] representation
  --> Classification head: Linear(768, 700) -> Sigmoid
  --> Output: {IAB-17: 0.97, IAB-17-12: 0.88, IAB-17-2: 0.83, ...}

Training: Full backpropagation through all 150M parameters
  - Learns domain-specific representations ("espn" token becomes sports-specialized)
  - Requires more data, longer training, GPU
  - Best absolute quality but more expensive to train and serve

Inference time: ~8-15ms (full encoder forward pass)
```

---

### Head-to-Head Comparison for "espn.com"

| Approach | IAB-17 (Sports) | IAB-17-12 (Football) | IAB-1 (Entertain.) | Latency | Quality Rank |
|---|---|---|---|---|---|
| TF-IDF + LR | 0.91 | 0.73 | 0.12 | 0.2ms | 5th |
| LLM Distilled | 0.96 | 0.81 | 0.22 | 3ms | 2nd |
| Embedding + Head | 0.89 | 0.68 | 0.08 | 5ms | 4th |
| SetFit (Tier-1) | 0.94 | N/A | 0.15 | 2ms | 3rd (Tier-1 only) |
| ModernBERT | 0.97 | 0.88 | 0.25 | 12ms | 1st |
| Claude Direct | 0.98 | 0.85 | 0.25 | 800ms | Ceiling |

The distilled student (Approach 2) achieves the best speed-quality tradeoff: nearly matching ModernBERT quality at 4x lower latency, and far exceeding TF-IDF quality at only 15x higher latency.

## Section 5: Extreme Multi-Label Strategy -- Handling 700 Classes

### The Challenge

Classifying into 700 categories is fundamentally different from classifying into 10. The challenges compound:

```
Standard classification (10 classes):
  - Each class has ~10% of examples
  - Model sees thousands of positives per class
  - Random baseline: 10% accuracy
  - Class confusion is manageable

Extreme multi-label (700 classes):
  - Average domain has 1-3 labels out of 700
  - For EACH class: 99.5% of examples are NEGATIVE
  - Many Tier-2 classes have <50 training examples
  - Similar classes are easily confused (IAB-17-12 Football vs IAB-17-44 Soccer)
  - Output is a 700-dim BINARY vector (multi-label, not mutually exclusive)
```

### Our Strategy: Hierarchical + Asymmetric Loss

We combine three techniques to handle the extreme class count:

**Technique 1: Hierarchical Classification (Tier-1 gates Tier-2)**

```
Instead of one flat 700-class classifier:

Stage 1: Tier-1 Classifier (23 classes)
  Input: domain embedding (384-dim)
  Output: which Tier-1 categories are active
  Example: espn.com --> {IAB-17: 0.96, IAB-1: 0.22}

Stage 2: Per-Tier-1 Sub-classifiers (only active categories run)
  For IAB-17 (Sports): run Sports sub-classifier (35 sub-categories)
    Output: {Football: 0.85, Baseball: 0.80, Basketball: 0.75, ...}
  For IAB-1 (Entertainment): run Entertainment sub-classifier (30 sub-categories)
    Output: {Television: 0.15, Celebrity: 0.10, ...}

Benefits:
  - Each sub-classifier handles only ~30 classes (tractable)
  - Only 2-3 sub-classifiers run per domain (fast)
  - Class imbalance is much less severe within each sub-classifier
  - Total effective parameters: 23 + (23 * ~30) = ~713 output neurons
    (same as flat, but structured)
```

**Technique 2: Asymmetric Loss (ASL)**

Standard binary cross-entropy treats positives and negatives equally. With 700 classes and 1-3 labels per domain, 99.5% of the signal is "this is NOT category X" -- the model drowns in easy negatives.

```
Standard BCE:
  L = -[y * log(p) + (1-y) * log(1-p)]
  Problem: 697 negative classes contribute overwhelming gradient

Asymmetric Loss:
  L_positive = -(1-p)^gamma_pos * log(p)        [gamma_pos = 0]
  L_negative = -(p_shifted)^gamma_neg * log(1-p) [gamma_neg = 4, shift = 0.05]

  - gamma_neg = 4: Massively down-weights EASY negatives
    (if model already predicts 0.01 for a negative, loss ~= 0)
  - shift = 0.05: Hard-threshold very easy negatives to zero loss
  - gamma_pos = 0: Never down-weight positives (they are precious)

Effect: Model focuses gradient on:
  1. Getting positives right (always weighted fully)
  2. Hard negatives (categories the model is confused about)
  Ignores: easy negatives (the 690+ categories that are obviously irrelevant)
```

**Technique 3: Label Embedding Similarity (Zero-Shot Capable)**

```
Core idea: encode both the INPUT (domain) and the LABELS (category names)
into the same embedding space, then measure similarity.

  domain_embedding = encode("espn.com")                     --> 384-dim
  label_embeddings = encode(["Sports", "Football", ...])    --> 700 x 384-dim

  scores = cosine_similarity(domain_embedding, label_embeddings)  --> 700-dim

Benefits:
  - Zero-shot: new categories can be added without retraining
    (just encode the new category name)
  - Naturally captures label semantics ("Football" is close to "Sports")
  - Works as a regularizer alongside the MLP head

Limitation:
  - Lower absolute accuracy than trained classification head
  - Best used as an auxiliary signal or for cold-start categories
```

### Per-Class Threshold Tuning

A global threshold of 0.5 is suboptimal for 700 heterogeneous classes:

```
  Class "Sports" (common):      optimal threshold = 0.6 (high precision needed)
  Class "Fly Fishing" (rare):   optimal threshold = 0.3 (maximize recall)
  Class "News" (ambiguous):     optimal threshold = 0.7 (reduce false positives)

Strategy:
  For each of 700 classes independently:
    1. Sweep thresholds [0.1, 0.2, ..., 0.9] on validation set
    2. Select threshold that maximizes F1 for THAT class
    3. Store per-class thresholds as a 700-dim vector

At inference:
  predictions = model(domain)               --> 700-dim probabilities
  active = predictions > per_class_thresholds  --> 700-dim binary
```

This typically improves macro-F1 by 5-15% over a global threshold, especially for rare classes.

## Section 6: Model Architecture Details

### Student Model Options

The student model has two components: an **encoder** (converts text to embeddings) and a **classification head** (converts embeddings to category probabilities).

```
ENCODER OPTIONS (ranked by speed):
+==============================================================================+
| Model              | Params | Dim  | Speed     | Quality | Context |
|====================|========|======|===========|=========|=========|
| all-MiniLM-L6-v2   | 22M    | 384  | 0.3ms     | Good    | 512     |
| e5-small-v2         | 33M    | 384  | 0.5ms     | Better  | 512     |
| gte-modernbert-base | 150M   | 768  | 1.5ms     | Best    | 8192    |
| e5-base-v2          | 110M   | 768  | 1.2ms     | Better+ | 512     |
| e5-large-v2         | 335M   | 1024 | 3.0ms     | Best+   | 512     |
+==============================================================================+

CLASSIFICATION HEAD (same for all encoders):
+==============================================================================+
| Architecture                              | Params | Speed  |
|===========================================|========|========|
| Linear(dim, 700) + Sigmoid                | ~550K  | 0.05ms | (simplest)
| Linear(dim, 256) + ReLU + Linear(256,700) | ~250K  | 0.08ms | (recommended)
| Hierarchical: Tier1(23) + Tier2(~30 each) | ~200K  | 0.10ms | (best for 700)
+==============================================================================+

RECOMMENDED CONFIGURATION (production):
  Encoder:  e5-small-v2 (33M params, 384-dim, 0.5ms)
  Head:     Linear(384, 256) + ReLU + Dropout(0.1) + Linear(256, 700) + Sigmoid
  Total:    ~33.3M params, ~0.6ms inference on CPU
  Quality:  95%+ agreement with Claude teacher
```

### Why E5-small as Default Encoder?

```
Decision matrix for our use case (domain names, short text, 700 classes):

  Input length: Domain names are SHORT (5-30 characters)
    --> Long context (8192 tokens) is unnecessary
    --> Smaller models are not disadvantaged by truncation

  Vocabulary: Domain names use common English words
    --> No need for specialized tokenizer
    --> All models handle this equally well

  Inference budget: <5ms per query
    --> Rules out e5-large (3ms encoder alone)
    --> MiniLM (0.3ms) and e5-small (0.5ms) both fit comfortably

  Quality: e5-small vs MiniLM on text classification benchmarks
    --> e5-small wins by 2-3% across MTEB classification tasks
    --> The 0.2ms extra latency is worth it

  Conclusion: e5-small-v2 is the Pareto-optimal choice.
```

### ModernBERT Architecture (for Approach 5)

ModernBERT (December 2024) modernizes the BERT architecture with several innovations:

```
ModernBERT-base (150M params):
  - 12 Transformer layers
  - 768 hidden dimension, 12 attention heads
  - Rotary Position Embeddings (RoPE) -- no fixed position limit
  - Flash Attention 2 -- 2-3x faster attention computation
  - Unpadding -- skips computation on padding tokens (variable-length efficiency)
  - 8192 token context window (vs. BERT's 512)
  - Trained on 2 TRILLION tokens (vs. BERT's 3.3 billion)
  - Training data includes code, web, and academic text

For classification:
  from transformers import ModernBertForSequenceClassification
  model = ModernBertForSequenceClassification.from_pretrained(
      "answerdotai/ModernBERT-base", num_labels=700,
      problem_type="multi_label_classification"
  )

vs. DeBERTa-v3 (previous best encoder):
  - Same quality on most tasks
  - 30-50% faster inference (Flash Attention + Unpadding)
  - 16x longer context (8192 vs 512)
  - Simpler architecture (no disentangled attention)
```

### SetFit Architecture (for Approach 4)

```
SetFit training pipeline (few-shot, no LLM needed):

Phase 1: Contrastive Fine-Tuning
  Given: 8-16 examples per Tier-1 category (184-368 total examples)

  Generate pairs:
    Positive: (espn.com, nfl.com)         label=1 (same category)
    Negative: (espn.com, zillow.com)      label=0 (different category)
    ~10K pairs total from 200-400 examples

  Fine-tune sentence-transformer with cosine similarity loss:
    loss = 1 - cos(embed(anchor), embed(positive))      for positive pairs
    loss = max(0, cos(embed(anchor), embed(negative)))  for negative pairs

  After fine-tuning: embeddings cluster by IAB category
    [Sports domains cluster together]
    [Travel domains cluster together]
    [Finance domains cluster together]

Phase 2: Classification Head Training
  Logistic Regression on fine-tuned embeddings:
    embed(domain) --> 384-dim --> LogReg --> 23-class probabilities

Total training time: 2-5 minutes on CPU
Total labeled data needed: ~200-400 examples (vs. 80,000 for full fine-tuning)

Limitation for our use case:
  - 700 classes requires 700 * (700-1) / 2 = 244,650 possible negative pairs
  - Contrastive training with this many pairs is slow and unstable
  - Works well for Tier-1 (23 classes: 253 pairs) but not Tier-2
```

## Section 7: Latency and Cost Budget

### Offline Phase Costs (One-Time)

```
DATA CORRECTION (Sonnet 4 via AWS Bedrock):
================================================================
Dataset size: 97K unique domains

Per-domain token usage:
  System prompt (categories + instructions): ~500 tokens
  User message (domain + metadata): ~100 tokens
  Response (JSON with category + keywords): ~50 tokens

Model: Claude Sonnet 4 (us.anthropic.claude-sonnet-4-20250514-v1:0)
Workers: 10 concurrent
Speed: ~7 domains/sec
Total time: ~4 hours
================================================================

TEACHER LABELING (Opus 4.6 via AWS Bedrock):
================================================================
Subset size: 16K+ stratified domains

Per-domain token usage:
  System prompt (IAB taxonomy + instructions): ~2,000 tokens
  User message (domain + metadata): ~200 tokens
  Response (JSON with soft probabilities): ~300 tokens

Model: Claude Opus 4.6 (global inference profile)
Workers: 5 concurrent
Speed: ~2.5 domains/sec
Total time: ~1 hour per 10K domains
================================================================

STUDENT TRAINING:
================================================================
Embedding generation (one-time, pre-computed):
  97K domains * E5-small (MPS): ~3 minutes at 600 domains/sec
  Output: 97K x 384 matrix (~143 MB as .npy)

MLP head training:
  Input: pre-computed embeddings (97K x 384)
  Training: 50 epochs, batch size 256
  Time: ~65 seconds on MPS
  Model size: 539 KB (135K parameters)
================================================================
```

### Online Phase Latency Budget

```
PRODUCTION INFERENCE (per request):
================================================================

TIER 1: LOOKUP TABLE (covers ~85% of traffic)
  Hash map lookup: domain_string --> IAB_vector
  Latency: <0.5ms
  Memory: 97K entries * ~600 bytes/entry = ~58MB (fits in L3 cache)

TIER 2: STUDENT MODEL (covers ~14% of traffic)
  Step                          | Latency (CPU)  | Latency (MPS/GPU)
  ------------------------------|----------------|-------------------
  Tokenization                  | 0.1ms          | 0.1ms
  E5-small forward pass         | 1.5ms          | 0.3ms
  MLP classification head       | 0.1ms          | 0.05ms
  Threshold + post-processing   | 0.05ms         | 0.05ms
  ------------------------------|----------------|-------------------
  TOTAL                         | ~2ms           | ~0.5ms

TIER 3: ASYNC LLM (covers ~1% of traffic)
  Queue domain for Claude API call (fire-and-forget)
  Return student prediction immediately
  Update lookup table when response arrives (~500ms-2s later)
  Latency impact on user: ZERO (async)

================================================================

COMPARISON WITH PREVIOUS PROJECT's TF-IDF PIPELINE
(github.com/nbatra/nlp-classifier-realtime):
  TF-IDF vectorization: ~0.1ms
  Logistic regression:  ~0.1ms
  TOTAL (TF-IDF):       ~0.2ms

  Student model (E5-small + MLP): ~2ms
  Student model on GPU:           ~0.5ms

  The student is 10x slower than TF-IDF on CPU but still well within
  the 10ms latency budget. On GPU, it matches TF-IDF speed.
  The quality improvement justifies the latency cost.

  For the LOOKUP TABLE path (85% of traffic): identical to TF-IDF speed.
================================================================
```

### Throughput at Scale

```
Scenario: 1M requests/second (AdTech scale)

Traffic distribution:
  850K req/s --> Tier 1 (lookup table): trivial, limited only by memory bandwidth
  140K req/s --> Tier 2 (student model):
    On CPU (8 cores): 8 * (1000ms / 2ms) = 4,000 req/s per machine
    Need: 140K / 4K = 35 CPU machines
    On GPU (A100): ~100K req/s per GPU (batched inference)
    Need: 2 GPUs
  10K req/s  --> Tier 3 (async): rate-limited by Claude API quota

This is identical to the previous project's scaling properties
(github.com/nbatra/nlp-classifier-realtime)
because the lookup table handles the vast majority of traffic.
```

## Section 8: Notebook Dependency Graph

### Two Pipelines

We run the full classification pipeline twice -- once on the original Kaggle labels (v1), and once on LLM-corrected labels (v2). This demonstrates the impact of label quality on model performance.

**Why two versions?** The Kaggle labels are noisy (49.7% disagreement with Sonnet/Opus). v2 uses Sonnet 4 to correct the ground-truth labels and generate English keywords for all domains, then re-runs the full pipeline. Keeping v1 allows direct comparison that proves label quality > model capacity.

```
notebooks/
  00_architecture_overview.ipynb          (this file -- shared reference)

  v1-kaggle/                              (original Kaggle labels)
    01_data_preparation_eda.ipynb         # Load Kaggle data, EDA, train/val/test split
    02_claude_teacher_labeling.ipynb      # Opus soft labels on 16K domains (8.4% coverage)
    03_embedding_generation.ipynb         # E5-small-v2 on raw text
    04_student_distillation_training.ipynb # MLP: 45.1% top-1 (noisy label ceiling)
    05_modernbert_finetune.ipynb          # ModernBERT: 61.3% top-1 (noisy labels)

  v2-llm-corrected/                       (Sonnet-corrected labels)
    01_data_correction_sonnet.ipynb       # Correct 97K domains (49.7% changed)
    02_eda.ipynb                          # EDA on corrected dataset
    03_embedding_generation.ipynb         # E5-small-v2 on enriched text (keywords)
    04_student_distillation_training.ipynb # MLP: 84.9% top-1 (337K params, 25 sec)
    05_modernbert_finetune.ipynb          # ModernBERT: 150M params (in progress)
    06_tfidf_baseline.ipynb               # TF-IDF + LinearSVC: 91.6% top-1 (BEST)
```

### Role of Each LLM

| Model | Role | What It Produces | Coverage | Key Metric |
|---|---|---|---|---|
| **Sonnet 4** | Label corrector + keyword generator | Hard tier1 label + English keywords per domain | All 97K domains | 49.7% labels changed |
| **Opus 4.6** | Soft label teacher | Probability distributions over 27 categories | 40,696 domains (32.9K in train) | Mean confidence 0.782 |

Sonnet fixes the ground truth and generates features. Opus provides the distillation signal. Downstream models learn from both.

### Data Flow Diagram (v2 Pipeline)

```
Kaggle Dataset (98K domains, noisy labels)
    |
    v
v1-kaggle/01 (Data Prep + EDA)
    |
    +---> v2/01 (Sonnet Correction) -------> data/corrected/{train,val,test}.parquet
    |         |                                  (corrected tier1 + English keywords)
    |         v
    |     v2/02 (EDA on corrected data)
    |         |
    |         +---> v2/03 (Embedding Generation) --> data/corrected/embeddings/*.npy
    |         |         |
    |         |         v
    |         |     v2/04 (MLP Distillation) --> models/student_mlp_v2_best.pt (84.9%)
    |         |
    |         +---> v2/05 (ModernBERT Fine-Tune) --> models/modernbert_v2_best/
    |         |
    |         +---> v2/06 (TF-IDF Baseline) --> models/tfidf_v2/ (91.6% -- BEST)
    |
    +---> v1-kaggle/02-05 (original pipeline for comparison)
              Results: MLP 45.1%, ModernBERT 61.3%
```

### Critical Path (v2)

```
v1/01 --> v2/01 --> v2/03 --> v2/04 --> compare results
  |         |         |         |
  Data   Correct   Embeds   Student
  ~10m   ~4hrs     ~3min    25 sec
        (async)

Independent branch:
v2/01 --> v2/06 (TF-IDF, no embeddings needed)
            |
          12 sec training
```

### Scripts (background processing)

```
scripts/
  run_teacher_labeling.py       # Opus 4.6: soft labels for 40K+ domains
                                # 209 checkpoint files, ~4hrs total
                                # Resumes automatically if interrupted
  run_sonnet_correction.py      # Sonnet 4: correct all 97K domains
                                # 524 checkpoint files, ~4hrs total
                                # Resumes automatically if interrupted
  check_status.py               # Monitor running LLM labeling processes
```

Both scripts checkpoint every 200 domains and resume automatically if interrupted. The Opus labeling produced 40,696 valid soft label vectors across 209 checkpoint files.

## Section 9: FAQ

**Q: Why not just call the LLM for every prediction in production?**

Three reasons that make it impossible at AdTech scale:
1. **Latency**: LLM takes 500-2000ms per request. Bid enrichment budget is <10ms.
2. **Cost**: At 1M QPS, calling the LLM costs ~$1M/day. The TF-IDF classifier costs ~$100/day in compute.
3. **Availability**: External API dependency in a real-time path creates an unacceptable failure mode. TF-IDF runs locally with no network dependency.

---

**Q: Why did TF-IDF beat the neural approaches?**

Because the LLM-generated keywords are effectively pre-classified features. When Sonnet writes "automotive, car dealership, vehicles" for a domain, TF-IDF converts "automotive" directly into a high-IDF feature that maps 1:1 to "Autos & Vehicles." Dense embeddings compress this explicit signal into 384 continuous dimensions where precision is lost.

This is specific to this pipeline. If the input were raw domain names without LLM-generated keywords, TF-IDF would perform poorly and embeddings would win.

---

**Q: Is the MLP distillation useless then?**

No. The MLP has one advantage: it works on raw text without requiring LLM-generated keywords. For **new domains** that have not been processed by Sonnet (no keywords available), you can still compute an E5 embedding from just the domain name and get 84.9% accuracy. TF-IDF without keywords would be much worse.

The production decision:
- Known domains (have keywords): TF-IDF (91.6%, 0.02ms)
- New domains (no keywords): MLP on raw embedding (84.9%, ~2ms)
- Very new domains (low confidence): Queue for LLM async labeling

---

**Q: How do we know the student actually learned correctly?**

We measure on a held-out test set (9,794 domains) that was never seen during training:
- TF-IDF: 91.7% top-1 (val: 91.6% -- 0.1pp difference, no overfitting)
- MLP: evaluated on same val set as TF-IDF for fair comparison
- Per-class analysis identifies failure modes (Sensitive Subjects: 0% F1 due to only 3 val samples)

---

**Q: What if the LLM makes mistakes in the labels?**

This is why the MLP uses a combined loss: `0.7 * KL(teacher) + 0.3 * weighted_BCE(hard_labels)`. The BCE term anchors the student to Sonnet-corrected labels. In practice:
- Sonnet-Opus agreement: 81.7% (both LLMs agree on the category)
- Where they disagree, Opus soft labels still provide useful distributional signal
- The 42.4% teacher coverage means most domains also have hard-label anchoring

---

**Q: Why two LLMs instead of one?**

Economics and task fit:
- **Sonnet 4** (label corrector): Straightforward classification + keyword extraction. Fast, cheap, covers all 97K domains. Total cost: ~$30-50 for the full dataset.
- **Opus 4.6** (teacher): Calibrated soft probability distributions. Slow, expensive, covers 40K strategic subset. Total cost: ~$100-150 for the full subset.

Using Opus for all 97K domains would cost 3x more and take 3x longer, with minimal quality gain for the hard-label task.

---

**Q: What is the minimum data needed for each approach?**

| Approach | Minimum Data | What It Needs |
|---|---|---|
| TF-IDF + LinearSVC | ~5K labeled with keywords | Keywords are the key feature |
| MLP distillation | ~5K soft labels + embeddings | Teacher labels drive quality |
| ModernBERT fine-tune | ~20K labeled examples | More data = less overfitting |

All approaches benefit from more data, but TF-IDF saturates earliest (diminishing returns past ~30K with this keyword quality).

---

**Q: How does this compare to the [previous project's](https://github.com/nbatra/nlp-classifier-realtime) TF-IDF pipeline?**

| Metric | Previous Project (raw TF-IDF) | This Project (LLM-enhanced TF-IDF) |
|---|---|---|
| Label quality | Noisy Kaggle labels (49.7% wrong) | Sonnet-corrected (81.7% LLM agreement) |
| Text features | Raw domain descriptions (multilingual, noisy) | LLM-generated English keywords (clean, semantic) |
| Accuracy | ~75% estimated (trained on noisy labels) | **91.6%** (trained on corrected labels + keywords) |
| Inference speed | ~0.2ms | 0.02ms (faster due to cleaner feature space) |
| Model complexity | Same (TF-IDF + linear classifier) | Same (TF-IDF + LinearSVC) |

The model is identical. The input quality is what changed. This is the core lesson of the project.

---

**Q: How often do we need to re-label / retrain?**

| Event | Action | Frequency |
|---|---|---|
| New domains discovered | TF-IDF classifies immediately (if keywords exist); else MLP on raw text | Real-time |
| Batch of new domains | Run Sonnet keyword generation, retrain TF-IDF | Weekly/monthly |
| IAB taxonomy update | Re-run Sonnet correction with new taxonomy; retrain | Yearly |
| Quality drift detected | Audit + re-label disagreements with LLM | Quarterly |

TF-IDF retraining takes 12 seconds. MLP retraining takes 25 seconds. The expensive part is LLM re-labeling (~4 hours for full dataset).

---

**Q: What hardware is needed?**

| Phase | Hardware | Time |
|---|---|---|
| Sonnet data correction | Any machine with internet (API calls, 10 workers) | ~4 hours |
| Opus teacher labeling | Any machine with internet (API calls, 5 workers) | ~4 hours |
| Embedding generation | CPU or MPS (Apple Silicon) | ~3 minutes |
| TF-IDF + LinearSVC training | CPU only | 12.5 seconds |
| MLP distillation training | CPU or MPS | 25 seconds |
| ModernBERT fine-tuning | MPS/CUDA recommended (150M params) | ~75 minutes |
| Production inference (TF-IDF) | CPU only | 0.02ms per domain |

The entire pipeline runs on a laptop. No GPU cluster required. The LLM calls are the only external dependency and they are fully async with checkpointing.

---

*This notebook is a reference document. No code executes here. Proceed to v2-llm-corrected/01 to begin the corrected pipeline, or v1-kaggle/01 for the baseline.*